# Lab 3.6 &mdash; Checkpointing

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; LangGraph: Stateful Agent Workflows**

### What you'll do
- Get persistence from one keyword argument
- Decide what a <code>thread_id</code> should be &mdash; it is an identity, not a token
- Read one snapshot with <code>get_state</code>, the whole trail with <code>get_state_history</code>

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All eight Module 3 labs work one case: leave requests in a small HR
> system. The rules are ordinary on purpose &mdash; the only new thing here is LangGraph.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-06")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# Leave requests in a small HR system. Ordinary rules on purpose: the only new thing in these
# eight labs is LangGraph. One flat dict -- no joins, no helpers, nothing to learn here.

REQUESTS = {
    "LV-5001": {"who": "Priya Nair",   "days":  3, "kind": "annual", "reason": "family wedding",
                "balance": 12, "manager": "Devi R."},
    "LV-5002": {"who": "Rahul Menon",  "days":  5, "kind": "annual", "reason": "",
                "balance":  3, "manager": "Devi R."},
    "LV-5003": {"who": "Anita Sharma", "days":  2, "kind": "annual", "reason": "moving house",
                "balance":  0, "manager": "Sam O."},
    "LV-5004": {"who": "Vikram Rao",   "days": 15, "kind": "annual", "reason": "sabbatical",
                "balance": 20, "manager": "Sam O."},
    "LV-5005": {"who": "Priya Nair",   "days":  1, "kind": "sick",   "reason": "flu",
                "balance": 12, "manager": "Devi R."},
}

# The handbook, as three numbers. Every routing decision in this module comes from these.
POLICY = {"manager_over_days": 2, "hr_over_days": 10, "max_clarifications": 2}

print(len(REQUESTS), "leave requests loaded")

## Concept

Everything so far vanished the moment `invoke()` returned. A **checkpointer** writes the state
after every node, so a run stops being an event and becomes a record.

```python
app = builder.compile(checkpointer=InMemorySaver())
app.invoke(state, {"configurable": {"thread_id": "LV-5002"}})
```

One argument, and four things become possible: **resume** a run later or in another process,
**inspect** it without re-running, **rewind** it, and **audit** it.

The fourth is the one nobody builds deliberately &mdash; it falls out of the other three, and it is
a stronger artefact than the model's own account of what it did, because it was recorded as it
happened rather than reconstructed afterwards.

`InMemorySaver` is the development one. `SqliteSaver` and `PostgresSaver` take the same argument;
only the constructor changes.

## Section 1 &mdash; Attach one, and choose the thread

`thread_id` names the thing being worked on. Runs that share one see each other's state; runs that
do not are strangers. Too broad and unrelated requests contaminate each other; too narrow and
"resume tomorrow" quietly starts from scratch.

In [ ]:
import uuid
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


class CaseState(TypedDict):
    request_id: str
    days: int
    decision: str
    notes: Annotated[list, add]


def summarise(state): return {"days": REQUESTS[state["request_id"]]["days"], "notes": ["summarise"]}
def decide(state):    return {"decision": "auto" if state["days"] <= POLICY["manager_over_days"]
                                          else "manager", "notes": ["decide"]}


def build_persistent():
    builder = StateGraph(CaseState)
    builder.add_node("summarise", summarise)
    builder.add_node("decide", decide)
    builder.add_edge(START, "summarise")
    builder.add_edge("summarise", "decide")
    builder.add_edge("decide", END)
    return builder.compile(checkpointer=BLANK)   # TODO: what makes this graph remember?


def thread_for(request_id: str) -> dict:
    """Two runs should share a thread when they are working the same ... what?"""
    per_request  = {"configurable": {"thread_id": request_id}}
    per_employee = {"configurable": {"thread_id": REQUESTS[request_id]["who"]}}
    per_run      = {"configurable": {"thread_id": str(uuid.uuid4())}}
    return BLANK        # TODO: which one lets you pick this request up again tomorrow?

In [ ]:
# --- Self-check: Section 1   (a real checkpointer, really invoked -- no model)
check("the same request twice lands on the same thread",
      lambda: thread_for("LV-5001") == thread_for("LV-5001"),
      "a fresh id per run would make 'resume tomorrow' impossible")
check("LV-5001 and LV-5005 are the same person and still do not share a thread",
      lambda: thread_for("LV-5001") != thread_for("LV-5005"),
      "keying on the employee would let one request overwrite the other")

def ran(rid):
    app = build_persistent()
    app.invoke({"request_id": rid, "notes": []}, thread_for(rid))
    return app

check("state is still there after invoke() returned",
      lambda: ran("LV-5005").get_state(thread_for("LV-5005")).values["decision"] == "auto")
score()

## Section 2 &mdash; Reading it back

Two calls, and the difference between them is the difference between a status page and an audit
trail. `get_state(config)` is one snapshot &mdash; where is this now? `get_state_history(config)`
is every checkpoint, newest first &mdash; what did it know, and when?

In [ ]:
def audit_trail(app, config) -> list:
    """What you hand someone who asks how this request reached its decision."""
    latest = [app.get_state(config)]
    every   = list(app.get_state_history(config))
    return BLANK        # TODO: which one answers "what was known at each step?"

In [ ]:
# --- Self-check: Section 2   (real checkpoint history off a real run)
def trail(rid="LV-5004"):
    return audit_trail(ran(rid), thread_for(rid))

check("the trail has a checkpoint per step, not just the final one",
      lambda: len(trail()) > 1,
      "get_state returns one snapshot; get_state_history returns all of them")
check("an earlier checkpoint exists in which no decision had been made yet",
      lambda: any(not s.values.get("decision") for s in trail()),
      "that is the point of a trail: it shows what was NOT yet known")
score()

## Watch it run &mdash; come back to it later

In [ ]:
app = guard(build_persistent)
cfg = guard(lambda: thread_for("LV-5004"))

if app is not None and cfg is not None:
    app.invoke({"request_id": "LV-5004", "notes": []}, cfg)

    snap = app.get_state(cfg)
    print("where is it now?  decision =", snap.values["decision"],
          " next =", snap.next or "(finished)")

    print("\nhow did it get there?")
    for s in reversed(list(app.get_state_history(cfg))):
        print(f"   next={str(s.next or ('END',)):22} notes={s.values.get('notes', [])}")

## Run it for real &mdash; the model's output lands in the record

`explain` asks the model for the sentence the employee reads. Run it, then look at the trail: the
output is *in* a checkpoint, at a known step, next to the state that produced it.

In [ ]:
if llm_ready():
    def explain(state):
        r = REQUESTS[state["request_id"]]
        text = ask(f'Tell {r["who"]} in one short sentence that their {r["days"]}-day leave '
                   f'request outcome is "{state["decision"]}". No greeting.',
                   system="You write brief internal HR messages.")
        return {"notes": [f"explain: {text.strip()[:100]}"]}

    b = StateGraph(CaseState)
    b.add_node("summarise", summarise); b.add_node("decide", decide); b.add_node("explain", explain)
    b.add_edge(START, "summarise"); b.add_edge("summarise", "decide")
    b.add_edge("decide", "explain"); b.add_edge("explain", END)
    live = b.compile(checkpointer=InMemorySaver())

    live_cfg = {"configurable": {"thread_id": "LV-5002"}}
    live.invoke({"request_id": "LV-5002", "notes": []}, live_cfg)

    for s in reversed(list(live.get_state_history(live_cfg))):
        print(f"   next={str(s.next or ('END',)):22} notes={s.values.get('notes', [])}")

### Read it

**Nobody wrote a log.** The history exists because the state is explicit and the checkpointer
saved it after every node &mdash; the return on the work done in Lab 3.1.

**It beats asking the model.** An agent asked to explain itself produces a plausible
reconstruction. The trail is what was actually true at each step, recorded before anyone knew the
outcome would matter.

In [ ]:
score()

## Your turn

1. Invoke the same thread again with a different `request_id` and read the state back. Did the
   notes reset? Should they have? That is the contamination a bad `thread_id` produces.
2. Fetch one old checkpoint with `app.get_state({"configurable": {"thread_id": ...,
   "checkpoint_id": ...}})` using an id from the history. That lookup is where an incident review
   starts &mdash; and where Lab 3.7's rewind starts.